In [1]:
import pandas as pd

df = pd.read_csv("../results/results.csv")

In [2]:
def generate_table(metric, resolution):

    # Convert resolution to string for matching
    resolution = str(resolution)

    table = (
        df[df["Resolution"].astype(str).str.contains(resolution, case=False, regex=False)]
        .sort_values(["Image", "Payload_Size(Bytes)"])
        [[
            "Image",
            "Payload",
            metric,
            "AES_Encrypt_Time(s)",
            "Dynamic_XOR_Time(s)",
            "Embedding_Time(s)",
            "Sender_Total_Time(s)",
            "Extraction_Time(s)",
            "XOR_Recovery_Time(s)",
            "AES_Decrypt_Time(s)",
            "Receiver_Total_Time(s)"
        ]]
        .reset_index(drop=True)
    )

    numeric_cols = table.select_dtypes(include="number").columns
    table[numeric_cols] = table[numeric_cols].round(6)

    return table

In [3]:
psnr_table = generate_table("PSNR", 1024)
psnr_table

,Image,Payload,PSNR,AES_Encrypt_Time(s),Dynamic_XOR_Time(s),Embedding_Time(s),Sender_Total_Time(s),Extraction_Time(s),XOR_Recovery_Time(s),AES_Decrypt_Time(s),Receiver_Total_Time(s)
0,aircraft_1024.png,100B.txt,86.653793,0.006260,0.000483,0.005163,0.011988,0.000430,0.000409,0.000842,0.001686
1,aircraft_1024.png,250B.txt,82.962810,0.000153,0.001180,0.002060,0.003525,0.000894,0.000787,0.000147,0.001832
2,aircraft_1024.png,500B.txt,80.102060,0.000125,0.000967,0.003934,0.005205,0.000843,0.000748,0.000155,0.001749
3,aircraft_1024.png,1KB.txt,76.886912,0.000189,0.004457,0.009567,0.014698,0.003348,0.002675,0.000141,0.006168
4,aircraft_1024.png,2KB.txt,73.951379,0.000205,0.007758,0.012187,0.021002,0.005835,0.006685,0.000168,0.012692
5,aircraft_1024.png,4KB.txt,70.963816,0.000208,0.009650,0.012676,0.023473,0.007556,0.006897,0.000125,0.014581
6,baboon_1024.png,100B.txt,86.713150,0.000089,0.000230,0.000605,0.000958,0.000244,0.000179,0.000078,0.000504
7,baboon_1024.png,250B.txt,82.842683,0.000135,0.000735,0.001112,0.002084,0.000380,0.000348,0.000066,0.000797
8,baboon_1024.png,500B.txt,80.117303,0.000078,0.000873,0.003145,0.004175,0.000768,0.000694,0.000072,0.001536
9,baboon_1024.png,1KB.txt,76.816979,0.000076,0.001781,0.003699,0.005707,0.001492,0.001421,0.000074,0.002990


In [4]:
def generate_table2(metric, resolution):
    """
    metric     : PSNR, MSE, NPCR, Entropy, SSIM, "Q-Index"
    resolution : 128, 256, 512, 1024
    """

    resolution = f"{resolution}x{resolution}"

    # Filter by resolution
    temp = df[df["Resolution"] == resolution].copy()

    # Clean image names
    temp["Image"] = (
        temp["Image"]
        .str.replace(f"_{resolution}", "", regex=False)
        .str.capitalize()
    )

    # Pivot table
    table = temp.pivot_table(
        index="Image",
        columns="Payload",
        values=metric,
        aggfunc="mean"
    )

    # Correct payload order
    payload_order = [
        "100B.txt",
        "250B.txt",
        "500B.txt",
        "1KB.txt",
        "2KB.txt",
        "4KB.txt",
    ]

    table = table.reindex(columns=payload_order)

    # Rename columns for publication
    table.columns = [
        col.replace(".txt", "").replace("K", " K").replace("B", " B").replace(" K B", " KB")
        for col in table.columns
    ]

    # Round
    table = table.round(4)

    # Cumulative average
    table.loc["Cumulative Average"] = table.mean()

    return table

In [5]:
generate_table2("PSNR", 1024)

,100 B,250 B,500 B,1 KB,2 KB,4 KB
Image,,,,,,
Aircraft_1024.png,86.65380,82.96280,80.10210,76.88690,73.95140,70.9638
Baboon_1024.png,86.71320,82.84270,80.11730,76.81700,73.92040,70.8949
House_1024.png,86.75320,83.20460,80.03950,76.91190,73.89950,70.9522
Pepper_1024.png,86.46160,83.13850,80.04810,76.95480,73.95930,70.9883
Cumulative Average,86.64545,83.03715,80.07675,76.89265,73.93265,70.9498


In [6]:
generate_table2("MSE", 1024)

,100 B,250 B,500 B,1 KB,2 KB,4 KB
Image,,,,,,
Aircraft_1024.png,0.0001,0.0003,0.0006,0.001300,0.0026,0.005200
Baboon_1024.png,0.0001,0.0003,0.0006,0.001400,0.0026,0.005300
House_1024.png,0.0001,0.0003,0.0006,0.001300,0.0026,0.005200
Pepper_1024.png,0.0001,0.0003,0.0006,0.001300,0.0026,0.005200
Cumulative Average,0.0001,0.0003,0.0006,0.001325,0.0026,0.005225


In [7]:
generate_table2("NPCR", 1024)

,100 B,250 B,500 B,1 KB,2 KB,4 KB
Image,,,,,,
Aircraft_1024.png,0.025000,0.056600,0.1135,0.232600,0.457900,0.915000
Baboon_1024.png,0.025700,0.054800,0.1135,0.231500,0.460900,0.914900
House_1024.png,0.024800,0.056900,0.1137,0.232800,0.460600,0.917200
Pepper_1024.png,0.023400,0.055400,0.1117,0.231600,0.454100,0.915600
Cumulative Average,0.024725,0.055925,0.1131,0.232125,0.458375,0.915675


In [7]:
generate_table2("Entropy", 1024)

,100 B,250 B,500 B,1 KB,2 KB,4 KB
Image,,,,,,
Aircraft_1024.png,6.676900,6.676900,6.676900,6.676900,6.67690,6.67690
Baboon_1024.png,7.753700,7.753700,7.753700,7.753700,7.75370,7.75370
House_1024.png,7.491500,7.491700,7.491800,7.492000,7.49210,7.49210
Pepper_1024.png,7.691600,7.691800,7.691900,7.691900,7.69190,7.69190
Cumulative Average,7.403425,7.403525,7.403575,7.403625,7.40365,7.40365


In [9]:
generate_table2("NPCR", 128)

,100 B,250 B,500 B,1 KB,2 KB,4 KB
Image,,,,,,
Aircraft_128.png,1.55030,3.595000,7.415800,14.93530,29.4373,58.813500
Baboon_128.png,1.62960,3.668200,7.330300,14.78270,29.2603,58.465600
House_128.png,1.62960,3.680400,7.153300,14.78880,29.6265,58.654800
Pepper_128.png,1.62350,3.710900,7.257100,14.70340,29.5715,58.819600
Cumulative Average,1.60825,3.663625,7.289125,14.80255,29.4739,58.688375


In [8]:
def generate_table3(metric, payload):
    """
    metric : str
        "PSNR", "MSE", "NPCR", "Entropy", "SSIM", "Q-Index"

    payload : str
        Example:
            "100B.txt"
            "250B.txt"
            "500B.txt"
            "1KB.txt"
            "2KB.txt"
            "4KB.txt"
    """

    # Filter selected payload
    temp = df[df["Payload"] == payload].copy()

    # Clean image names
    temp["Image"] = (
        temp["Image"]
        .str.replace(".png", "", regex=False)
        .str.replace(r"_\d+$", "", regex=True)
        .str.title()
    )

    # Make resolution labels prettier
    temp["Resolution"] = temp["Resolution"].replace({
        "1024x1024": "(1024 × 1024)",
        "512x512": "(512 × 512)",
        "256x256": "(256 × 256)",
        "128x128": "(128 × 128)"
    })

    # Create table
    table = temp.pivot_table(
        index="Image",
        columns="Resolution",
        values=metric,
        aggfunc="mean"
    )

    # Order resolutions
    resolution_order = [
        "(1024 × 1024)",
        "(512 × 512)",
        "(256 × 256)",
        "(128 × 128)"
    ]

    table = table.reindex(columns=resolution_order)

    # Round
    table = table.round(6)

    # Cumulative average
    table.loc["Cumulative Average"] = table.mean()

    return table

In [9]:
generate_table3("PSNR", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,80.102060,73.997505,68.087023,61.911762
Baboon,80.117303,74.021021,68.024065,61.964842
House,80.039478,74.038204,67.923929,61.838523
Pepper,80.048056,74.057616,67.882008,61.994852
Cumulative Average,80.076724,74.028587,67.979256,61.927495


In [10]:
generate_table3("Entropy", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,6.676907,6.663923,6.676778,6.670931
Baboon,7.753692,7.762442,7.739406,7.739333
House,7.491760,7.486112,7.489011,7.492683
Pepper,7.691888,7.671181,7.694615,7.698576
Cumulative Average,7.403562,7.395915,7.399952,7.400381


In [15]:
generate_table3("NPCR", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,0.113487,0.454712,1.812744,7.415771
Baboon,0.113487,0.451279,1.828003,7.330322
House,0.113678,0.458527,1.818848,7.153320
Pepper,0.111675,0.447845,1.814270,7.257080
Cumulative Average,0.113082,0.453091,1.818466,7.289123


In [11]:
generate_table3("MSE", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,0.000635,0.002590,0.010101,0.041870
Baboon,0.000633,0.002576,0.010249,0.041361
House,0.000644,0.002566,0.010488,0.042582
Pepper,0.000643,0.002555,0.010590,0.041077
Cumulative Average,0.000639,0.002572,0.010357,0.041723


In [12]:
generate_table3("SSIM", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,1.0,0.999999,0.999997,0.999987
Baboon,1.0,1.000000,0.999998,0.999993
House,1.0,1.000000,0.999998,0.999992
Pepper,1.0,1.000000,0.999998,0.999992
Cumulative Average,1.0,1.000000,0.999998,0.999991


In [13]:
generate_table3("Q-Index", "500B.txt")

Resolution,(1024 × 1024),(512 × 512),(256 × 256),(128 × 128)
Image,,,,
Aircraft,1.0,0.999999,0.999997,0.999987
Baboon,1.0,1.000000,0.999998,0.999993
House,1.0,1.000000,0.999998,0.999992
Pepper,1.0,1.000000,0.999998,0.999992
Cumulative Average,1.0,1.000000,0.999998,0.999991
